In [1]:
import requests
import pandas as pd
import numpy as np
import pandas as pd
import joblib
model = joblib.load('models/la_xgboost.pkl')

# Open-Meteo forecast API (different endpoint from historical)
url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 34.0549,
    "longitude": -118.2426,
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min", 
        "precipitation_sum",
        "windspeed_10m_max",
        "winddirection_10m_dominant",
        "dewpoint_2m_mean",
        "relative_humidity_2m_mean",
        "pressure_msl_mean",
        "cloudcover_mean",
        "shortwave_radiation_sum"
    ],
    "timezone": "America/Los_Angeles",
    "start_date": "2026-04-22",
    "end_date": "2026-04-25"
}

response = requests.get(url, params=params)
forecast_df = pd.DataFrame(response.json()['daily'])
print(forecast_df)

         time  temperature_2m_max  temperature_2m_min  precipitation_sum  \
0  2026-04-22                21.2                10.3                0.0   
1  2026-04-23                22.6                11.9                0.0   
2  2026-04-24                24.2                16.3                0.0   
3  2026-04-25                18.9                15.0                0.0   

   windspeed_10m_max  winddirection_10m_dominant  dewpoint_2m_mean  \
0               20.2                         270               7.4   
1               22.4                         222               7.2   
2               18.9                         184               8.4   
3               30.0                         221               8.0   

   relative_humidity_2m_mean  pressure_msl_mean  cloudcover_mean  \
0                         61             1019.0                2   
1                         53             1014.7               76   
2                         51             1011.7               57

In [2]:
# Recent actual AQI values (from AirNow)
recent_aqi_actual = [47, 46, 52, 67, 100, 77, 54]  # Apr 14 - Apr 20
today_aqi = 34  # April 21
today_ozone_dominant = 1  # April 21 2026: ozone dominant per AirNow

# All 8 values for rolling calculations
all_recent = recent_aqi_actual + [today_aqi]

# Calculate rolling features from recent actual data
rolling_7d_mean = np.mean(all_recent[-7:])
rolling_7d_std = np.std(all_recent[-7:])

# Days since rain — check forecast, April 10 has rain
# Look back at recent data, was there rain recently?
days_since_rain_apr22 = 1  # approximate, update on April 21 with real value

print(f"Rolling 7d mean: {rolling_7d_mean:.1f}")
print(f"Rolling 7d std: {rolling_7d_std:.1f}")

Rolling 7d mean: 61.4
Rolling 7d std: 20.3


In [7]:
predictions = []
lag_1 = today_aqi

for i, (_, row) in enumerate(forecast_df.iterrows()):
    features = {
        'temperature_2m_max': row['temperature_2m_max'],
        'temperature_2m_min': row['temperature_2m_min'],
        'precipitation_sum': row['precipitation_sum'],
        'windspeed_10m_max': row['windspeed_10m_max'],
        'winddirection_10m_dominant': row['winddirection_10m_dominant'],
        'dewpoint_2m_mean': row['dewpoint_2m_mean'],
        'relative_humidity_2m_mean': row['relative_humidity_2m_mean'],
        'pressure_msl_mean': row['pressure_msl_mean'],
        'cloudcover_mean': row['cloudcover_mean'],
        'shortwave_radiation_sum': row['shortwave_radiation_sum'],
        'month': 4,
        'day_of_week': pd.to_datetime(row['time']).dayofweek,
        'is_weekend': int(pd.to_datetime(row['time']).dayofweek >= 5),
        'aqi_lag_1': lag_1,
        'aqi_lag_2': all_recent[-1] if len(predictions) == 0 else predictions[-1]['predicted_aqi'],
        'aqi_lag_3': (
            all_recent[-2] if len(predictions) == 0
            else all_recent[-1] if len(predictions) == 1
            else predictions[-2]['predicted_aqi']
        ),
        'aqi_lag_7': all_recent[i] if i < len(all_recent) else predictions[i-7]['predicted_aqi'],
        'aqi_lag_14': all_recent[max(0, i-7)],
        'ozone_dominant_lag1': today_ozone_dominant,
        'had_rain': int(row['precipitation_sum'] >= 1),
        'days_since_rain': max(0, days_since_rain_apr22 + i),
        'aqi_rolling_7d_mean': np.mean(all_recent[-7:]),
        'aqi_rolling_7d_std': np.std(all_recent[-7:])
    }

    X_pred = pd.DataFrame([features])
    pred = model.predict(X_pred)[0]
    predictions.append({'date': row['time'], 'predicted_aqi': pred})
    lag_1 = pred

for p in predictions:
    print(f"{p['date']}: {p['predicted_aqi']:.2f}")

2026-04-22: 46.53
2026-04-23: 47.61
2026-04-24: 58.91
2026-04-25: 49.67


In [4]:
for p in predictions:
    print(f"{p['date']}: {p['predicted_aqi']:.1f}")

2026-04-22: 46.5
2026-04-23: 47.6
2026-04-24: 58.9
2026-04-25: 49.7


In [5]:
print(len(predictions))
print(predictions)

4
[{'date': '2026-04-22', 'predicted_aqi': np.float32(46.530964)}, {'date': '2026-04-23', 'predicted_aqi': np.float32(47.61465)}, {'date': '2026-04-24', 'predicted_aqi': np.float32(58.910156)}, {'date': '2026-04-25', 'predicted_aqi': np.float32(49.666126)}]


In [6]:
print(forecast_df.shape)
print(forecast_df.head())

(4, 11)
         time  temperature_2m_max  temperature_2m_min  precipitation_sum  \
0  2026-04-22                21.2                10.3                0.0   
1  2026-04-23                22.6                11.9                0.0   
2  2026-04-24                24.2                16.3                0.0   
3  2026-04-25                18.9                15.0                0.0   

   windspeed_10m_max  winddirection_10m_dominant  dewpoint_2m_mean  \
0               20.2                         270               7.4   
1               22.4                         222               7.2   
2               18.9                         184               8.4   
3               30.0                         221               8.0   

   relative_humidity_2m_mean  pressure_msl_mean  cloudcover_mean  \
0                         61             1019.0                2   
1                         53             1014.7               76   
2                         51             1011.7         